[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/Physics/blob/main/Simulaciones_Jupyter/Metodos_Matematicos_%28MTH%29/Simulacion_y_Machine_Learning_%28physics.comp-ph%29/MTH-06_PINN_Dinamica_Fluidos_Burgers.ipynb)

# [MTH-06] PINN: Ecuación de Burgers (Choque de Fluidos)

Nota: Este emulador utiliza `scipy.optimize` y derivadas simbólicas en NumPy para ilustrar el mecanismo subyacente de Autograd (PyTorch/JAX) en un entorno puramente didáctico de PINNs sin dependencias de tensores GPU.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

# EMULADOR PINN: Ecuación no lineal de Burgers 1D
# u_t + u * u_x - nu * u_xx = 0

nu = 0.01 / np.pi # Viscosidad cinemática
np.random.seed(42)

# Collocation points (x, t) en el dominio interior
N_f = 200
x_f = np.random.uniform(-1, 1, N_f)
t_f = np.random.uniform(0, 1, N_f)

# Arquitectura de Red Neuronal Simulada: 1 capa oculta
# u(x,t) = w2 * tanh(w1_x * x + w1_t * t + b1) + b2
def nn_forward(params, x, t):
    w1_x, w1_t, b1, w2, b2 = params[0], params[1], params[2], params[3], params[4]
    z = w1_x * x + w1_t * t + b1
    return w2 * np.tanh(z) + b2

# Derivadas automáticas emuladas (Analíticas de la red)
def nn_derivs(params, x, t):
    w1_x, w1_t, b1, w2, b2 = params[0], params[1], params[2], params[3], params[4]
    z = w1_x * x + w1_t * t + b1
    sech2 = 1.0 - np.tanh(z)**2
    u = w2 * np.tanh(z) + b2
    u_t = w2 * w1_t * sech2
    u_x = w2 * w1_x * sech2
    # u_xx
    tanh_z = np.tanh(z)
    u_xx = w2 * w1_x**2 * (-2 * tanh_z * sech2)
    return u, u_t, u_x, u_xx

# Función de Pérdida (Loss)
def loss_fn(params):
    # 1. Physical Loss (Burgers PDE)
    u, u_t, u_x, u_xx = nn_derivs(params, x_f, t_f)
    f_res = u_t + u * u_x - nu * u_xx
    mse_f = np.mean(f_res**2)
    
    # 2. Boundary Condition Loss (u(-1, t) = u(1, t) = 0)
    u_bc1 = nn_forward(params, -1.0 * np.ones_like(t_f), t_f)
    u_bc2 = nn_forward(params,  1.0 * np.ones_like(t_f), t_f)
    mse_bc = np.mean(u_bc1**2) + np.mean(u_bc2**2)
    
    # 3. Initial Condition Loss (u(x, 0) = -sin(pi * x))
    u_ic = nn_forward(params, x_f, np.zeros_like(x_f))
    target_ic = -np.sin(np.pi * x_f)
    mse_ic = np.mean((u_ic - target_ic)**2)
    
    return mse_f + 10 * mse_bc + 10 * mse_ic # Pesos para forzar BC/IC

# Entrenamiento de la PINN mediante BFGS
initial_params = np.random.randn(5) * 0.1
print("Iniciando entrenamiento PINN (Optimizando Loss Físico)...")
res = minimize(loss_fn, initial_params, method='L-BFGS-B', options={'maxiter': 500})
trained_params = res.x
print(f"Entrenamiento completado. Loss Final: {res.fun:.6f}")

# Visualización del resultado predicho por la IA Físicamente informada
x_test = np.linspace(-1, 1, 100)
t_test = np.linspace(0, 1, 100)
X, T = np.meshgrid(x_test, t_test)
U_pred = nn_forward(trained_params, X, T)

plt.figure(figsize=(10, 6))
plt.contourf(T, X, U_pred, 50, cmap='seismic')
plt.colorbar(label='Velocidad $u(x,t)$')
plt.title("Onda de Choque de Burgers resuelta por PINN")
plt.xlabel("Tiempo (t)")
plt.ylabel("Posición (x)")
plt.show()

